# Exportes para negocio, por grupo de consumo

Último paso de la corrida. No calcula nada nuevo: toma las tres salidas del mes
(pronóstico a 6 meses, lista de caída y lista de riesgo de fuga), les pone los nombres del
glosario (zona, clase de servicio, tipo de medidor y de lectura, promedio semestral, valor
facturado) y las parte por **grupo de consumo**:

| Grupo | Criterio (mediana de los últimos 12 meses) |
|---|---|
| Intermitente | ≤ 10 kWh/mes o la mitad o más de los meses en cero |
| Pequeño | 10 a 500 kWh/mes |
| Mediano | 500 a 5.000 kWh/mes |
| Grande | 5.000 kWh/mes o más |
| Sin historia suficiente | menos de 6 meses válidos (no se pronostica) |

Todo queda en `11_exportes_negocio`, con un índice (`indice_exportes.csv`) de qué archivo
tiene qué, cuántas filas y de qué corte es. La página web descarga estos mismos archivos.

In [ ]:
# ============================================================
# 1. RUTAS
# ============================================================
from pathlib import Path
import os
import numpy as np
import pandas as pd
from IPython.display import display

from utilidades_glosario import (
    enriquecer_glosario, agregar_atributos_a_lista, atributos_ultimo_mes, grupo_desde_perfil,
    nombre_archivo_grupo, GRUPOS_CONSUMO_ORDEN, GRUPO_CONSUMO_DESCRIPCION, COLUMNAS_GLOSARIO_SALIDA,
)

BASE_DIR = Path(os.environ.get("EBSA_DATOS", r"C:\Users\Home\Documents\Datos_Ebsa"))
PROCESADO_DIR = BASE_DIR / "01_historico_procesado"
MODELO_DIR = BASE_DIR / "04_pronostico" / "modelo_final"
GESTION_DIR = BASE_DIR / "07_gestion_caida"
FUGA_DIR = BASE_DIR / "10_riesgo_fuga"
EXP_DIR = BASE_DIR / "11_exportes_negocio"
for sub in ["pronostico_6_meses", "lista_caida", "riesgo_fuga"]:
    (EXP_DIR / sub).mkdir(parents=True, exist_ok=True)

RUTA_PRONOSTICO = MODELO_DIR / "predicciones_segmentadas_optimizadas_6_meses.parquet"
RUTA_PERFILES = MODELO_DIR / "perfiles_consumidores_corte_final.parquet"
RUTA_GERENCIAL = GESTION_DIR / "gestion_caida_gerencial.csv"
RUTA_FUGA = FUGA_DIR / "lista_riesgo_fuga_gerencial.csv"
RUTA_FUGA_TODOS = FUGA_DIR / "riesgo_fuga_clientes.csv"
RUTA_INDICE = EXP_DIR / "indice_exportes.csv"

indice = []

def exportar_por_grupo(df, carpeta, prefijo, corte, col_grupo="grupo_consumo"):
    """Un CSV por grupo de consumo + uno con todos. Devuelve el resumen para el índice."""
    filas = []
    grupos = [g for g in GRUPOS_CONSUMO_ORDEN if g in df[col_grupo].unique()] + \
             sorted(set(df[col_grupo].dropna().unique()) - set(GRUPOS_CONSUMO_ORDEN))
    for g in grupos:
        sub = df[df[col_grupo].eq(g)]
        ruta = EXP_DIR / carpeta / f"{prefijo}_{nombre_archivo_grupo(g)}.csv"
        sub.to_csv(ruta, index=False, encoding="utf-8-sig")
        filas.append({"producto": carpeta, "grupo_consumo": g, "archivo": str(ruta.relative_to(BASE_DIR)),
                      "clientes": len(sub), "fecha_corte": corte, "criterio_grupo": GRUPO_CONSUMO_DESCRIPCION.get(g, "")})
    ruta = EXP_DIR / carpeta / f"{prefijo}_todos_los_grupos.csv"
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    filas.append({"producto": carpeta, "grupo_consumo": "Todos", "archivo": str(ruta.relative_to(BASE_DIR)),
                  "clientes": len(df), "fecha_corte": corte, "criterio_grupo": ""})
    # limpiar archivos de grupos que ya no existen (de corridas anteriores)
    vigentes = {Path(f["archivo"]).name for f in filas}
    for viejo in (EXP_DIR / carpeta).glob(f"{prefijo}_*.csv"):
        if viejo.name not in vigentes:
            viejo.unlink()
    return filas

print("Exportes ->", EXP_DIR)


In [ ]:
# ============================================================
# 2. PRONÓSTICO A 6 MESES POR GRUPO
# ============================================================
if RUTA_PRONOSTICO.exists():
    pron = pd.read_parquet(RUTA_PRONOSTICO, engine="pyarrow")
    pron["NIU"] = pron["NIU"].astype("string").str.strip()
    corte = pd.to_datetime(pron["fecha_corte"]).max().to_period("M").to_timestamp()   # corte urbano = mes de la corrida
    atributos = atributos_ultimo_mes(PROCESADO_DIR, meses_atras=3, hasta=corte)
    pron = agregar_atributos_a_lista(pron, atributos, None)      # ya trae 'perfil' -> grupo_consumo
    cols_pred = [c for c in pron.columns if c.startswith("pred_") or c.startswith("fecha_pred_")]
    cols = (["NIU", "grupo_consumo", "zona", "ciclo", "zona_nombre", "clase_servicio", "clase_servicio_nombre", "estrato",
             "consumo_ultimo_mes_kwh", "consumo_promedio_semestral_kwh"]
            + cols_pred + ["promedio_pred_3m_kwh", "promedio_pred_6m_kwh", "tarifa_aplicada_kwh", "valor_facturado_mes",
                           "valor_facturado_origen", "tipo_medidor_nombre", "tipo_lectura_nombre", "regimen_actual",
                           "fecha_corte", "fecha_corte_modelo", "modo", "nota"])
    pron = pron[[c for c in cols if c in pron.columns]]
    # Nota para los rurales: sus primeros meses pronosticados ya pasaron pero no tienen
    # lectura trimestral todavía; el valor real reemplaza al pronosticado cuando llegue.
    if "zona" in pron.columns:
        corte_urb = pd.to_datetime(pron.loc[pron["zona"].eq("URBANO"), "fecha_corte"]).max() if pron["zona"].eq("URBANO").any() else corte
        def nota_fila(r):
            if r["zona"] != "RURAL":
                return ""
            meses = [str(pd.Timestamp(r[f"fecha_pred_{h}m"]))[:7] for h in range(1, 7)
                     if pd.Timestamp(r[f"fecha_pred_{h}m"]) <= corte_urb]
            return ("meses " + ", ".join(meses) + ": pronosticado, a la espera de la lectura trimestral") if meses else ""
        pron["nota"] = pron.apply(nota_fila, axis=1)
    for c in [c for c in pron.columns if c.startswith("fecha_")]:
        pron[c] = pd.to_datetime(pron[c]).dt.strftime("%Y-%m")
    for c in [c for c in pron.columns if c.endswith("_kwh")]:
        pron[c] = pd.to_numeric(pron[c], errors="coerce").round(1)
    indice += exportar_por_grupo(pron, "pronostico_6_meses", "pronostico_6_meses", f"{corte:%Y-%m}")
    print(f"Pronóstico: {len(pron):,} clientes, corte {corte:%Y-%m}")
    display(pron["grupo_consumo"].value_counts().rename("clientes").to_frame().T)
    del atributos
else:
    print("⚠ No hay pronóstico en", RUTA_PRONOSTICO)


In [ ]:
# ============================================================
# 3. LISTA DE CAÍDA POR GRUPO
# ============================================================
if RUTA_GERENCIAL.exists():
    caida = pd.read_csv(RUTA_GERENCIAL, dtype={"NIU": "string"}, encoding="utf-8-sig")
    corte = str(pd.to_datetime(caida["fecha_corte"]).max())[:7] if "fecha_corte" in caida.columns and len(caida) else ""
    if "grupo_consumo" not in caida.columns:       # lista de una versión anterior: completar
        atributos = atributos_ultimo_mes(PROCESADO_DIR, meses_atras=3, hasta=pd.Timestamp(corte + "-01") if corte else None)
        perfiles = pd.read_parquet(RUTA_PERFILES, columns=["NIU", "perfil"], engine="pyarrow") if RUTA_PERFILES.exists() else None
        caida = agregar_atributos_a_lista(caida, atributos, perfiles)
    indice += exportar_por_grupo(caida, "lista_caida", "lista_caida", corte)
    print(f"Lista de caída: {len(caida):,} clientes, corte {corte}")
    display(caida["grupo_consumo"].value_counts().rename("clientes").to_frame().T)
else:
    print("⚠ No hay lista de caída en", RUTA_GERENCIAL)


In [ ]:
# ============================================================
# 4. RIESGO DE FUGA POR GRUPO
# ============================================================
if RUTA_FUGA.exists():
    fuga = pd.read_csv(RUTA_FUGA, dtype={"NIU": "string"}, encoding="utf-8-sig")
    corte = str(pd.to_datetime(fuga["fecha_corte"]).max())[:7] if len(fuga) else ""
    indice += exportar_por_grupo(fuga, "riesgo_fuga", "riesgo_fuga_alto_y_medio", corte)
    print(f"Riesgo de fuga (ALTO + MEDIO): {len(fuga):,} clientes, corte {corte}")
    display(fuga["grupo_consumo"].value_counts().rename("clientes").to_frame().T)
    # también todos los puntuados, por grupo, para quien quiera la base completa
    if RUTA_FUGA_TODOS.exists():
        todos = pd.read_csv(RUTA_FUGA_TODOS, dtype={"NIU": "string"}, encoding="utf-8-sig")
        indice += exportar_por_grupo(todos, "riesgo_fuga", "riesgo_fuga_todos_los_puntuados", corte)
else:
    print("⚠ No hay lista de riesgo de fuga en", RUTA_FUGA)


In [ ]:
# ============================================================
# 5. ÍNDICE
# ============================================================
indice_df = pd.DataFrame(indice)
indice_df.to_csv(RUTA_INDICE, index=False, encoding="utf-8-sig")
print("EXPORTES PARA NEGOCIO — TERMINADO")
print("=" * 78)
display(indice_df)
print("Índice:", RUTA_INDICE)
